# Task 2 – Global Economic Prospects (June 2026): Structure-Aware RAG
**Advanced RAG Hands-on Challenge**

| Step | What happens |
|------|--------------|
| 1 | Load 200-page GEP PDF (page-by-page) |
| 2 | Detect chapter & section boundaries; attach to page metadata |
| 3 | Chunk with header injection — every chunk is self-contained |
| 4 | Embed chunks → FAISS (Gemini Embedding 2) |
| 5 | Build retriever (k=5) |
| 6 | Wire LLM with strict document-only system prompt |
| 7 | Validate with required questions; confirm no cross-section confusion |

**Key challenge vs Task 1:** This document is 200 pages with distinct chapters and regional
sections that all discuss similar concepts (growth, outlook, risks). Naive chunking causes
the retriever to mix up facts from different regions or chapters.

**Solution:** *Section-aware chunking with header injection* — every chunk is tagged with
`chapter` and `section` metadata, and the section label is prepended into the chunk text
so the LLM always knows where the content came from.

> **Stack:** Python · LangChain (LCEL) · Google Gemini · FAISS

---
## 0 · Install dependencies

In [ ]:
%pip install -q \
    langchain \
    langchain-google-genai \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    python-dotenv

---
## 0b · Configure API key

In [ ]:
import os, warnings, time, re
warnings.filterwarnings('ignore')

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

if not os.getenv('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = 'AQ.Ab8RN6LZMzf8GK5UsxOgWaYetgIbmegtM13F8KVoUbNoCAiLOg'

print('API key ready.')

---
## Step 1 · Load the PDF

`PyPDFLoader` extracts text page-by-page and attaches `page` number metadata.
The 200-page report loads in a few seconds; we preview a few pages to verify extraction quality.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = '/home/rahul-chokshi/Downloads/Rag Use cases-20261014T155740Z-1-001/Rag Use cases/GEP-Jun-2026.pdf'

loader = PyPDFLoader(PDF_PATH)
pages  = loader.load()

total_chars = sum(len(p.page_content) for p in pages)
print(f'Loaded {len(pages)} pages — {total_chars:,} total characters')
print(f'Average characters per page: {total_chars // len(pages)}')

# Preview a content-rich page
print(f'\n--- Page 24 preview (Summary section) ---')
print(pages[24].page_content[:600])

---
## Step 2 · Detect structure and annotate pages

**Why this matters:** The GEP has 4 chapters and ~12 distinct regional/topical sections,
all discussing GDP growth, forecasts, and risks. Without structure tags, a query about
"South Asia growth" might retrieve chunks from the Sub-Saharan Africa or East Asia
sections — they score similarly under cosine similarity.

**Approach:**
1. Hard-code chapter page ranges (verified by inspecting running headers)
2. Scan each page for known section headings to track current section
3. Write `chapter` and `section` into every page's `.metadata`

| Chapter | Pages | Title |
|---------|-------|-------|
| 1 | 22–67 | Global Outlook |
| 2 | 68–105 | Regional Outlooks |
| 3 | 106–141 | Sovereign Debt |
| 4 | 142–191 | Fiscal Policy |
| — | 192+ | Statistical Appendix |

In [ ]:
# ── Chapter page ranges (0-indexed, inclusive) ────────────────────────────────
CHAPTER_RANGES = [
    (22,  67,  'Chapter 1 – Global Outlook'),
    (68,  105, 'Chapter 2 – Regional Outlooks'),
    (106, 141, 'Chapter 3 – Sovereign Debt in Developing Economies'),
    (142, 191, 'Chapter 4 – Fiscal Policy'),
    (192, 999, 'Statistical Appendix'),
]

# ── Section headings mapped to their START page ──────────────────────────────
# Each entry: (start_page, section_label)
# A section is active from its start page until the next entry.
SECTION_BOUNDARIES = [
    (0,   'Front Matter'),
    (22,  'Chapter 1 – Global Outlook'),
    (24,  'Ch1 – Summary'),
    (34,  'Ch1 – Global Context and Trade'),
    (35,  'Ch1 – Commodity Markets'),
    (36,  'Ch1 – Global Inflation and Financial Developments'),
    (40,  'Ch1 – Major Economies: Recent Developments and Outlook'),
    (68,  'Chapter 2 – Regional Outlooks (Overview)'),
    (80,  'Ch2 – East Asia and Pacific'),
    (83,  'Ch2 – Europe and Central Asia'),
    (87,  'Ch2 – Latin America and the Caribbean'),
    (91,  'Ch2 – Middle East and North Africa'),
    (95,  'Ch2 – South Asia'),
    (98,  'Ch2 – Sub-Saharan Africa'),
    (106, 'Chapter 3 – Sovereign Debt in Developing Economies'),
    (142, 'Chapter 4 – Fiscal Policy in Developing Economies'),
    (163, 'Statistical Appendix'),
]

def get_chapter(page_num):
    for start, end, label in CHAPTER_RANGES:
        if start <= page_num <= end:
            return label
    return 'Front Matter / Other'

def get_section(page_num):
    """Return the section label active on this page."""
    current = SECTION_BOUNDARIES[0][1]
    for start, label in SECTION_BOUNDARIES:
        if page_num >= start:
            current = label
        else:
            break
    return current

# Annotate every page's metadata
for page in pages:
    pg = page.metadata.get('page', 0)
    page.metadata['chapter'] = get_chapter(pg)
    page.metadata['section'] = get_section(pg)

# Spot-check
print('Structure annotation spot-check:')
for check_pg in [24, 50, 80, 95, 98, 110, 150, 165]:
    m = pages[check_pg].metadata
    print(f'  Page {check_pg:>3} | chapter: {m["chapter"]} | section: {m["section"]}')

---
## Step 3 · Structure-aware chunking with header injection

**Key technique — header injection:** Each chunk's text is prefixed with
`[Chapter X – Section Name]` before being embedded. This means the vector for a
South Asia chunk explicitly contains "South Asia" semantic signal, making it much
less likely to be confused with a similar-sounding Sub-Saharan Africa chunk.

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `chunk_size` | 900 chars | Slightly larger to keep more section context together |
| `chunk_overlap` | 150 chars | Catch facts that span chunk boundaries |
| Header prefix | `[Chapter \| Section]` | Injected into chunk text for semantic anchoring |

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Skip front matter / blank pages for embedding
SKIP_BEFORE_PAGE = 22

content_pages = [p for p in pages
                 if p.metadata.get('page', 0) >= SKIP_BEFORE_PAGE
                 and len(p.page_content.strip()) > 100]

print(f'Content pages to chunk: {len(content_pages)} (skipped front matter < page {SKIP_BEFORE_PAGE})')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

raw_chunks = splitter.split_documents(content_pages)

# ── Header injection ─────────────────────────────────────────────────────────
# Prepend section context to every chunk so the embedding carries location signal
enriched_chunks = []
for chunk in raw_chunks:
    chapter = chunk.metadata.get('chapter', '')
    section = chunk.metadata.get('section', '')
    header  = f'[{chapter} | {section}]\n'
    enriched = Document(
        page_content=header + chunk.page_content,
        metadata=chunk.metadata,
    )
    enriched_chunks.append(enriched)

print(f'Total chunks after splitting: {len(raw_chunks)}')
print(f'Chunks with injected headers: {len(enriched_chunks)}')
print(f'Average chunk length: {sum(len(c.page_content) for c in enriched_chunks)//len(enriched_chunks)} chars')

# Show a sample from each major section
print('\n--- Sample chunk from South Asia section ---')
south_asia = [c for c in enriched_chunks if 'South Asia' in c.metadata.get('section', '')]
if south_asia:
    print(south_asia[0].page_content[:400])

print('\n--- Sample chunk from Sub-Saharan Africa section ---')
ssa = [c for c in enriched_chunks if 'Sub-Saharan' in c.metadata.get('section', '')]
if ssa:
    print(ssa[0].page_content[:400])

---
## Step 4 · Embeddings → FAISS vector store

With ~500 enriched chunks from 178 content pages, we embed in batches of 50
to stay within the free-tier rate limit (100 requests/minute).
Each batch gets a 70-second pause between it and the next.

In [ ]:
import time as _time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-2')

# Batch-pause strategy: embed 45 chunks, then pause 75s so the
# 60s rolling quota window fully resets before the next batch.
# Within a batch: 1.2s between calls → 50 req/min (50% margin under 100/min limit).
BATCH_SIZE  = 45
BATCH_PAUSE = 75
CHUNK_DELAY = 1.2

def _embed_one(text, emb, max_retries=5):
    """Embed one text, retrying on rate-limit errors."""
    for attempt in range(max_retries):
        try:
            return emb.embed_documents([text])[0]
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg or 'quota' in msg.lower():
                m = re.search(r'retry in (\d+)', msg)
                suggested = int(m.group(1)) if m else 60
                wait = max(suggested + 10, 70)
                print(f'\n  [429] waiting {wait}s (attempt {attempt+1}/{max_retries})...', flush=True)
                _time.sleep(wait)
            else:
                raise
    raise RuntimeError(f'Embedding failed after {max_retries} retries.')

texts  = [c.page_content for c in enriched_chunks]
metas  = [c.metadata     for c in enriched_chunks]
total  = len(texts)
total_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

print(f'Embedding {total} chunks in batches of {BATCH_SIZE}')
print(f'Within-batch delay: {CHUNK_DELAY}s | Pause between batches: {BATCH_PAUSE}s')
print(f'Estimated time: ~{(BATCH_SIZE * CHUNK_DELAY + BATCH_PAUSE) * total_batches / 60:.1f} min\n')

t0      = _time.time()
vectors = []

for batch_idx in range(0, total, BATCH_SIZE):
    batch      = texts[batch_idx : batch_idx + BATCH_SIZE]
    batch_num  = batch_idx // BATCH_SIZE + 1
    print(f'Batch {batch_num}/{total_batches}: chunks {batch_idx}–{batch_idx+len(batch)-1}', flush=True)

    for j, text in enumerate(batch):
        vectors.append(_embed_one(text, embeddings))
        if j < len(batch) - 1:
            _time.sleep(CHUNK_DELAY)

    done = batch_idx + len(batch)
    print(f'  → {done}/{total} embedded', flush=True)

    if done < total:
        print(f'  → pausing {BATCH_PAUSE}s before next batch...', flush=True)
        _time.sleep(BATCH_PAUSE)

# Build FAISS from pre-computed vectors (no extra API requests)
text_vec_pairs = list(zip(texts, vectors))
vector_store   = FAISS.from_embeddings(text_vec_pairs, embeddings, metadatas=metas)

elapsed = _time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min  |  {vector_store.index.ntotal} vectors in FAISS index')
_time.sleep(3)
print('Ready.')

---
## Step 5 · Retriever

We use **k=5** (vs k=4 in Task 1) because:
- The document is 200 pages — relevant content for a single question may span more chunks
- Section answers (e.g., South Asia outlook) often have multiple paragraphs spread across 2-3 pages

We test retrieval with a region-specific query to confirm section isolation.

In [ ]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5},
)

def test_retrieval(query):
    docs = retriever.invoke(query)
    print(f'Query: "{query}"')
    print(f'Retrieved {len(docs)} chunks:')
    for i, doc in enumerate(docs, 1):
        pg  = doc.metadata.get('page', '?')
        sec = doc.metadata.get('section', '?')
        print(f'  [{i}] Page {pg} | Section: {sec}')
        print(f'       {doc.page_content[doc.page_content.find(chr(10))+1:doc.page_content.find(chr(10))+1+120].strip()}...')
    print()
    return docs

# Test 1: Should retrieve ONLY South Asia chunks, not Sub-Saharan Africa
_ = test_retrieval('What is the growth outlook for South Asia?')

# Test 2: Should retrieve Chapter 3 chunks about sovereign debt
_ = test_retrieval('What does the report say about sovereign debt in developing economies?')

---
## Step 6 · LLM + document-only system prompt

Same strict grounding rules as Task 1, with one addition:
- The system prompt instructs the model to **cite the chapter and section** in its answer
  (the injected headers make this information available in the context).

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash', temperature=0)

SYSTEM_PROMPT = """You are a precise document assistant for the World Bank Global Economic Prospects report (June 2026).
Answer ONLY from the document excerpts in <context>.

Rules — no exceptions:
1. Base your answer exclusively on <context>. Do NOT use training knowledge.
2. If the answer is not present in <context>, respond with exactly:
   "I don't have that information."
3. Do not guess, extrapolate, or paraphrase beyond what is stated.
4. Each context excerpt is labeled with [Chapter | Section] — use this label to cite the
   source chapter and section in your answer (e.g., "According to Chapter 2 – South Asia...").
5. Never mix facts from different regions or chapters. If the question asks about South Asia,
   only use South Asia excerpts; do not use Sub-Saharan Africa or East Asia excerpts.

<context>
{context}
</context>"""

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    MessagesPlaceholder('chat_history'),
    ('human', '{question}'),
])

answer_chain = qa_prompt | llm | StrOutputParser()

def format_docs(docs):
    parts = []
    for d in docs:
        pg  = d.metadata.get('page', '?')
        sec = d.metadata.get('section', '')
        parts.append(f'[Page {pg} | {sec}]\n{d.page_content}')
    return '\n\n---\n\n'.join(parts)

def call_with_retry(fn, payload, label='call', retries=3, delay=2):
    for attempt in range(retries):
        try:
            return fn(payload)
        except Exception as e:
            if attempt < retries - 1:
                wait = delay * (attempt + 1)
                print(f'[{label} retry {attempt+1}] {e} — waiting {wait}s...')
                time.sleep(wait)
            else:
                raise

def ask(question, chat_history=None):
    """Retrieve relevant chunks and generate a grounded, section-cited answer."""
    if chat_history is None:
        chat_history = []
    docs    = call_with_retry(retriever.invoke, question, label='retrieval')
    context = format_docs(docs)
    payload = {'question': question, 'chat_history': chat_history, 'context': context}
    return call_with_retry(answer_chain.invoke, payload, label='LLM'), docs

print('Chain ready.')

---
## Step 7 · Validation — Required Assignment Questions

**Pass/fail bar:** The system must find the *right chapter or section* and must NOT mix up
facts from different regions or chapters.

| # | Question | Expected source |
|---|----------|-----------------|
| 1 | Growth outlook for South Asia | Ch2 – South Asia section |
| 2 | What Chapter 3 says about sovereign debt | Ch3 – Sovereign Debt |
| 3 | Main risks to global outlook (executive summary) | Ch1 – Summary |
| 4 | East Asia and Pacific growth forecast | Ch2 – East Asia and Pacific |
| 5 | Sub-Saharan Africa outlook | Ch2 – Sub-Saharan Africa |

In [ ]:
SAMPLE_QUESTIONS = [
    # --- Assignment required questions (pass/fail bar) ---
    ('REQUIRED', 'What is the growth outlook for South Asia in this report?'),
    ('REQUIRED', 'What does Chapter 3 say about sovereign debt in developing economies?'),
    ('REQUIRED', 'According to the executive summary, what are the main risks to the global outlook?'),
    # --- Section isolation checks ---
    ('ISOLATION', 'What is the GDP growth forecast for East Asia and Pacific in 2026?'),
    ('ISOLATION', 'What is the outlook for Sub-Saharan Africa?'),
]

SEPARATOR = '=' * 72

results = []
for label, question in SAMPLE_QUESTIONS:
    print(SEPARATOR)
    print(f'[{label}] {question}')
    print(SEPARATOR)

    answer, source_docs = ask(question)
    results.append({'label': label, 'question': question, 'answer': answer, 'sources': source_docs})

    print(f'Answer:\n{answer}')
    print()
    print('Source chunks retrieved:')
    sections_retrieved = set()
    for j, doc in enumerate(source_docs, 1):
        pg  = doc.metadata.get('page', '?')
        sec = doc.metadata.get('section', '?')
        sections_retrieved.add(sec)
        print(f'  [{j}] Page {pg} | {sec}')
        print(f'       {doc.page_content[50:170].strip()}...')
    print(f'\n  Distinct sections retrieved: {sections_retrieved}')
    print()
    time.sleep(1)

---
## Section isolation test · No cross-contamination check

Ask about two regions back-to-back and verify the answers don't share facts.

In [ ]:
isolation_pairs = [
    ('What is the GDP growth forecast for South Asia in 2026?',
     'What is the GDP growth forecast for Sub-Saharan Africa in 2026?'),
    ('What are the main risks for East Asia and Pacific?',
     'What are the main risks for Europe and Central Asia?'),
]

print('--- Section isolation: verifying no cross-contamination ---\n')
for q1, q2 in isolation_pairs:
    a1, docs1 = ask(q1)
    a2, docs2 = ask(q2)
    secs1 = {d.metadata.get('section') for d in docs1}
    secs2 = {d.metadata.get('section') for d in docs2}
    overlap = secs1 & secs2

    print(f'Q1: {q1}')
    print(f'   Sections: {secs1}')
    print(f'Q2: {q2}')
    print(f'   Sections: {secs2}')
    status = 'PASS (no overlap)' if not overlap else f'WARN — overlapping sections: {overlap}'
    print(f'   Isolation check: {status}')
    print()
    time.sleep(1)

---
## Out-of-scope test · Hallucination guard

In [ ]:
out_of_scope = [
    'What is the GDP growth forecast for the United States in 2030?',
    'What does the IMF say about inflation in 2024?',
]

print('--- Out-of-scope / hallucination guard ---')
for q in out_of_scope:
    answer, _ = ask(q)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print()

---
## Summary

| Component | Choice | Notes |
|-----------|--------|-------|
| **Document loader** | `PyPDFLoader` | Page-level metadata |
| **Structure detection** | Hard-coded chapter ranges + section scan | Verified against running headers |
| **Chunker** | `RecursiveCharacterTextSplitter` | 900 chars / 150 overlap |
| **Header injection** | `[Chapter \| Section]` prefix on every chunk | Prevents cross-section retrieval confusion |
| **Embeddings** | `models/gemini-embedding-2` | Dense vectors via Google AI Studio |
| **Vector store** | FAISS (in-memory) | No external DB required |
| **Retriever** | Similarity search, k=5 | Larger k for longer document |
| **LLM** | `gemini-3.6-flash`, temperature=0 | Deterministic answers |
| **Chain** | LCEL (`\|` operator) | No deprecated `langchain.chains` |
| **Grounding** | Strict system prompt + section citation rule | Fallback: *"I don't have that information."* |

**Key difference from Task 1 (baseline):** Section-aware chunking with header injection
ensures that regions and chapters are semantically anchored in the embedding space,
preventing the retriever from mixing up facts across similar-sounding sections.

**Next:** Task 3 will handle tables and structured numeric data — a different challenge
where the facts exist as rows and columns rather than narrative prose.